# 02 · SSL pretraining (SimCLR, P_ecg mid, seed 0)

**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

Pretrains the xresnet1d50 encoder on PTB-XL folds 1–8 **without labels** (`docs/ssl_protocol_v1.md`).
The run checkpoints to Drive every 5 epochs. If Colab disconnects, open the notebook again and **Run all**:
the timing cell is skipped and pretraining resumes from the last checkpoint.

Do not use *File → Save a copy in GitHub*.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile, json
PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'
CACHE_DRIVE, RUNS, SSL = f'{PROJECT}/cache', f'{PROJECT}/runs', f'{PROJECT}/ssl'
CACHE = '/content/cache'                      # local copy: much faster reads than Drive
REPO = '/content/robust-ecg-ssl'
os.makedirs(SSL, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Salako07/robust-ecg-ssl.git {REPO}
!git -C {REPO} pull -q && git -C {REPO} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
7e0aead (HEAD -> main, origin/main, origin/HEAD) Colab: retry cache copy when the Drive mount drops (Failure 5)
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# copy the cache to local disk once per session (~1-3 min)
os.makedirs(CACHE, exist_ok=True)
for f in os.listdir(CACHE_DRIVE):
    s, d = f'{CACHE_DRIVE}/{f}', f'{CACHE}/{f}'
    if os.path.isfile(s) and (not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s)):
        shutil.copy(s, d)
!ls -la {CACHE}

total 2228208
drwxr-xr-x 2 root root       4096 Sep 27 07:12 .
drwxr-xr-x 1 root root       4096 Sep 27 07:11 ..
-rw------- 1 root root        629 Sep 27 07:12 norm_ptbxl_folds1-8.json
-rw------- 1 root root    6594879 Sep 27 07:12 ptbxl_meta.csv
-rw------- 1 root root        589 Sep 27 07:12 ptbxl_statements.json
-rw------- 1 root root 1046352128 Sep 27 07:12 ptbxl_X.npy
-rw------- 1 root root       9720 Sep 27 07:12 scp_statements.csv
-rw------- 1 root root     991543 Sep 27 07:12 sph_meta.csv
-rw------- 1 root root 1227696128 Sep 27 07:12 sph_X.npy


## Timing check
30 steps at batch 512 and a projection for 300 epochs. Skipped once the run has started.
If it reports **out of memory**, stop and paste the error in the chat (do not change the batch size here:
batch size is part of the frozen protocol and any change has to be logged).

In [3]:
RUN = 'SSL_ecg-mid_s0'
if not os.path.exists(f'{SSL}/{RUN}/ckpt_last.pt') and not os.path.exists(f'{SSL}/{RUN}/done.json'):
    !python {REPO}/scripts/pretrain_simclr.py --policy ecg --strength mid --seed 0 --cache {CACHE} --runs {SSL} --max-steps 30
    !nvidia-smi --query-gpu=memory.used --format=csv
else:
    print('run already started, timing skipped')

SSL_ecg-mid_s0: 17418 unlabelled records, 34 steps/epoch, 300 epochs, device=cuda
/content/robust-ecg-ssl/scripts/pretrain_simclr.py:105: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"({total} steps); last loss {float(loss):.3f}, top-1 {float(acc):.3f}")
timing: 0.204 s/step -> 0.58 h for 300 epochs (10200 steps); last loss 2.784, top-1 0.643
memory.used [MiB]
0 MiB


## Pretraining
Prints one line per epoch: NT-Xent loss, in-batch retrieval top-1 (chance ≈ 0.001), learning rate, minutes.

In [4]:
!python {REPO}/scripts/pretrain_simclr.py --policy ecg --strength mid --seed 0 --cache {CACHE} --runs {SSL} --workers 2

SSL_ecg-mid_s0: 17418 unlabelled records, 34 steps/epoch, 300 epochs, device=cuda
epoch 1/300  loss 3.6341  top-1 0.554  lr 1.03e-04  0.1 min
epoch 2/300  loss 2.1301  top-1 0.775  lr 2.03e-04  0.2 min
epoch 3/300  loss 1.5287  top-1 0.889  lr 3.03e-04  0.2 min
epoch 4/300  loss 1.1648  top-1 0.932  lr 4.03e-04  0.3 min
epoch 5/300  loss 0.8904  top-1 0.955  lr 5.03e-04  0.4 min
epoch 6/300  loss 0.6898  top-1 0.972  lr 6.03e-04  0.5 min
epoch 7/300  loss 0.5657  top-1 0.980  lr 7.03e-04  0.5 min
epoch 8/300  loss 0.4775  top-1 0.987  lr 8.03e-04  0.6 min
epoch 9/300  loss 0.3991  top-1 0.992  lr 9.03e-04  0.7 min
epoch 10/300  loss 0.3547  top-1 0.994  lr 1.00e-03  0.8 min
epoch 11/300  loss 0.3124  top-1 0.996  lr 1.00e-03  0.9 min
epoch 12/300  loss 0.2883  top-1 0.996  lr 1.00e-03  0.9 min
epoch 13/300  loss 0.2679  top-1 0.997  lr 1.00e-03  1.0 min
epoch 14/300  loss 0.2502  top-1 0.997  lr 1.00e-03  1.1 min
epoch 15/300  loss 0.2371  top-1 0.998  lr 9.99e-04  1.2 min
epoch 16/300

In [5]:
import pandas as pd
print(json.dumps(json.load(open(f'{SSL}/{RUN}/done.json')), indent=1))
log = pd.read_csv(f'{SSL}/{RUN}/log.csv')
print(log.iloc[[0, 1, 4, 9, 19, 49, 99, 149, 199, 249, -1]].round(4).to_string(index=False))

{
 "run_id": "SSL_ecg-mid_s0",
 "policy": "ecg",
 "strength": "mid",
 "seed": 0,
 "epochs": 300,
 "bs": 512,
 "tau": 0.1,
 "final_loss": 0.09444579986088417,
 "final_top1": 1.0,
 "minutes_last_session": 23.576749547322592,
 "encoder_sha256": "0cc027b0c1e31fe3045ccb78c9e544b704b7e21e520de3a84d307fcf029a94b7"
}
 epoch   loss   top1     lr  minutes
     1 3.6341 0.5544 0.0001   0.0964
     2 2.1301 0.7749 0.0002   0.1675
     5 0.8904 0.9552 0.0005   0.3948
    10 0.3547 0.9942 0.0010   0.7765
    20 0.1999 0.9984 0.0010   1.5635
    50 0.1373 0.9997 0.0010   3.9504
   100 0.1129 1.0000 0.0008   7.8903
   150 0.1049 0.9996 0.0005  11.8391
   200 0.0985 0.9999 0.0003  15.7605
   250 0.0962 0.9999 0.0001  19.6854
   300 0.0944 1.0000 0.0000  23.5767


## First matched pair at 100% labels (seed 0)
Two runs of the core grid, ~6 min each: S1 (random init + P_ecg) and C1 (this encoder + P_ecg).
Together with S0 they give the first single-seed look at C1 − S1. **One seed, no confidence interval:
not evidence for or against any hypothesis.**

In [6]:
!python {REPO}/scripts/train.py --arm S1 --budget 1.0 --seed 0 --cache {CACHE} --runs {RUNS} --workers 2
!python {REPO}/scripts/train.py --arm C1 --budget 1.0 --seed 0 --cache {CACHE} --runs {RUNS} --workers 2 --pretrained {SSL}/{RUN}/encoder.pt
keys = ['test_macro_auroc', 'best_val_macro_auroc', 'best_step', 'test_e3_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation']
rows = {}
for rid in ['S0_b1_s0_lik0', 'S1_b1_s0_lik0', 'C1_b1_s0_lik0']:
    p = f'{RUNS}/{rid}/done.json'
    if os.path.exists(p):
        r = json.load(open(p)); rows[rid] = {k: r[k] for k in keys} | {k[10:]: r[k] for k in r if k.startswith('sph_auroc_')}
print(pd.DataFrame(rows).round(4).to_string())

S1_b1_s0_lik0: train=17418 records, 6800 steps, eval every 136, device=cuda
/content/robust-ecg-ssl/scripts/train.py:137: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log.append(dict(step=step, loss=float(loss), val_macro_auroc=auc, minutes=(time.time() - t0) / 60))
step 136/6800  loss 0.6732  val macro-AUROC 0.5559 (best 0.5559, 71 labels)  0.1 min
step 272/6800  loss 0.2855  val macro-AUROC 0.6154 (best 0.6154, 71 labels)  0.2 min
step 408/6800  loss 0.1473  val macro-AUROC 0.6853 (best 0.6853, 71 labels)  0.3 min
step 544/6800  loss 0.0984  val macro-AUROC 0.7698 (best 0.7698, 71 labels)  0.5 min
step 680/6800  loss 0.0874  val macro-AUROC 0.7991 (best 0.7991, 71 labels)  0.6 min
step 816/6800  loss 0.0850  val macro-AUROC 0.8153 (best 0.8153, 71 labels)  0.7 min
step 952/6800  loss 0.0876  

## Export results for the repository
Creates `results_export.zip` on Drive with every finished run's small files (fine-tuning runs and SSL runs;
`encoder.pt` and checkpoints are left out, their hashes are in the JSON files). **Download it and attach it in the chat.**

In [7]:
out = f'{PROJECT}/results_export.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, arc in ((RUNS, 'runs'), (SSL, 'ssl')):
        for f in glob.glob(f'{root}/*/*'):
            if f.endswith(('config.json', 'log.csv', 'done.json', 'predictions.npz')) and \
               os.path.exists(os.path.join(os.path.dirname(f), 'done.json')):
                z.write(f, os.path.join(arc, os.path.relpath(f, root)))
    for reg, arc in ((f'{RUNS}/registry.csv', 'runs/registry.csv'), (f'{SSL}/registry_ssl.csv', 'ssl/registry_ssl.csv')):
        if os.path.exists(reg):
            z.write(reg, arc)
print(out, round(os.path.getsize(out) / 1e6, 2), 'MB')
print(zipfile.ZipFile(out).namelist())

/content/drive/MyDrive/robust-ecg-ssl/results_export.zip 2.14 MB
['runs/S0_b1_s0_lik0/config.json', 'runs/S0_b1_s0_lik0/predictions.npz', 'runs/S0_b1_s0_lik0/log.csv', 'runs/S0_b1_s0_lik0/done.json', 'runs/S1_b1_s0_lik0/config.json', 'runs/S1_b1_s0_lik0/predictions.npz', 'runs/S1_b1_s0_lik0/log.csv', 'runs/S1_b1_s0_lik0/done.json', 'runs/C1_b1_s0_lik0/config.json', 'runs/C1_b1_s0_lik0/predictions.npz', 'runs/C1_b1_s0_lik0/log.csv', 'runs/C1_b1_s0_lik0/done.json', 'ssl/SSL_ecg-mid_s0/config.json', 'ssl/SSL_ecg-mid_s0/log.csv', 'ssl/SSL_ecg-mid_s0/done.json', 'runs/registry.csv', 'ssl/registry_ssl.csv']


**Paste the outputs of the timing cell, the summary cell and the matched-pair cell into the chat, and attach the zip.**